# 17 · Nâng baseline `R0` — eval-time, ensemble, multi-view

Hiện thực nhánh **1a** và **2a** của kế hoạch (`sleepy-puzzling-harbor.md`).

## Vì sao notebook này tồn tại

nb12–16 đã đóng nhánh **reweight loss** — WASL, deferred-WASL, DB-Loss, hybrid, temporal
targeting đều không qua cổng `+0.002`. nb16 chỉ ra lý do: can thiệp chỉ dịch **0.84%** khối
lượng gradient âm.

Và nb16 để lại ba con số định hình mọi việc sau đó:

| số | hệ quả |
|---|---|
| `ep0` (chưa train) macro `0.4355` ≈ `ep2` `0.4365`, CI chứa 0 | partial FT gần như không mua được gì |
| `ep2 → ep8` macro `−0.0068` | 8 epoch ở `lr = 1e-5` **làm hại**, đơn điệu |
| CI lấy mẫu val trên Rare-3 mAP `±0.0018`, **dùng chung mọi seed** | cải tiến `< ~0.004` chỉ nhắm Rare-3 là **không đo được** |

Nên notebook này **không** train backbone. Nó khai thác ba thứ chưa từng được đụng tới:

| | chưa từng kiểm |
|---|---|
| `T1.2` | eval dùng `CenterCrop(518)` từ ảnh 560 — viền `21px` mỗi bên bị vứt |
| `T1.3` | ensemble nhiều checkpoint |
| `V1–V3` | `pool_study` cố định `W_F = 0.7 / W_L = 0.3`, và **mean** trong mỗi view |

## ⚠ Ràng buộc phương pháp — đọc trước khi chạy

**Không tinh chỉnh trên VAL rồi báo cáo trên VAL.** Với `CI = ±0.0018`, thử 10 biến thể rồi
lấy số tốt nhất sẽ **chắc chắn** cho một con số lạc quan giả.

Mọi tham số **học từ dữ liệu** (trọng số pooling, pooler) phải **cross-fit 5-fold theo
`subject_id` trong val**: fit trên 4 fold, dự đoán fold còn lại, gộp toàn bộ dự đoán
held-out rồi mới tính AP.

Ensemble bằng **trung bình đơn thuần** không có tham số fit nên **không** cần cross-fit — nhưng
việc *chọn* tổ hợp nào theo val thì có. Nên tổ hợp được **khoá trước** ở `C0`.

**TEST không bao giờ chạm.**

## Thứ tự chạy

```
C0 -> C1 -> C2 -> C3      CPU thuan, KHONG can stage anh, cho so ngay
C4 -> C5                  GPU ~40 phut (per-image + ablation transform)
C6 -> C7                  CPU, multi-view + tong ket
```

## C0 · Config + prereg

Tổ hợp ensemble và quy tắc trung bình **khoá ở đây**, trước khi thấy bất kỳ con số nào.

Tổ hợp chính `{ep0, R0 ep1, R0 ep2}` chọn theo tiêu chí có sẵn từ nb16: đó là **đúng ba điểm
nằm trong dung sai macro `0.0015`** của checkpoint rule §9 — không phải chọn theo hiệu năng.

Ensemble **báo cáo tách riêng** khỏi single-model. Hai câu hỏi khác nhau; trộn vào nhau sẽ che
mất việc từng đòn bẩy đơn lẻ có tác dụng hay không.

In [3]:
import os, json, math, time, copy, pickle, random, shutil, tarfile, contextlib, hashlib
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score, roc_auc_score

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
LOCAL_IMG = os.environ.get('LOCAL_IMG', '/content/cxr_local')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
BB_CKPT   = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')
H5_CKPT   = os.path.join(DATA_DIR, 'token_fusion', 'checkpoints', 'H5_s42_best.pt')
NB16_DIR  = os.path.join(DATA_DIR, 'label_reliability')          # nguon prediction da luu
NB16_PRED = f'{NB16_DIR}/preds'
NB16_SNAP = f'{NB16_DIR}/snapshots'
SC_DIR    = os.path.join(DATA_DIR, 'baseline_opt')               # thu muc RIENG, khong ghi de nb16
for _d in (SC_DIR, f'{SC_DIR}/metrics', f'{SC_DIR}/preds'): os.makedirs(_d, exist_ok=True)

SEED   = 42
W_F, W_L = 0.7, 0.3            # pooling hien hanh cua nb11-16, dung lam MOC
N_BOOT = 2000                  # y het nb16 -> hai bang doc chung duoc
K_FOLD = 5                     # cross-fit theo subject_id
RARE3  = ['Pleural_Other', 'Fracture', 'Lung_Lesion']
WORST3_FIXED = ['Fracture', 'Pleural_Other', 'Enlarged_Cardiomediastinum']
MODEL_SIZE, STORE_SIZE = 518, 560

# ---------------- KHOA TRUOC (prereg) ----------------
ENS_PRIMARY = ['EP0', 'R0_ep1', 'R0_ep2']      # dung 3 diem trong dung sai macro 0.0015 cua §9
ENS_RULE    = 'mean_prob'                      # trung binh XAC SUAT; rank/logit chi la exploratory
REF_MODEL   = 'R0_ep2'                         # model tham chieu cho phan GPU
TRANSFORMS  = ['center518', 'resize518', 'tta5crop']   # center518 = hien hanh
PREREG = dict(
    notebook='17_r0_baseline_optimization.ipynb',
    plan='sleepy-puzzling-harbor.md — nhanh 1a + 2a',
    goal='nang baseline tuyet doi KHONG train backbone',
    ensemble=dict(primary=ENS_PRIMARY, rule=ENS_RULE,
                  chosen_by='3 diem nam trong dung sai macro 0.0015 cua ckpt rule §9 (nb16), '
                            'KHONG chon theo hieu nang',
                  reported='TACH RIENG khoi single-model'),
    reference_model=REF_MODEL, transforms=TRANSFORMS,
    cross_fit=f'{K_FOLD}-fold theo subject_id TRONG val; moi tham so fit tu du lieu deu phai qua day',
    no_cross_fit_needed='ensemble trung binh don thuan (khong co tham so fit)',
    bootstrap=dict(n=N_BOOT, cluster='subject_id', ap='ap_grouped verbatim nb16 C15'),
    baseline='nb16 R0_s42_ep2 study-level, pooling W_F=0.7/W_L=0.3',
    no_test_access=True)
json.dump(PREREG, open(f'{SC_DIR}/preregistration.json', 'w'), indent=1)

try:
    import torch
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
except Exception:
    dev = 'cpu'
print('device:', dev)
print(f'ensemble KHOA: {ENS_PRIMARY} | rule {ENS_RULE} | bao cao TACH RIENG')
print(f'cross-fit: {K_FOLD}-fold theo subject_id trong val')
print(f'nguon prediction nb16: {NB16_PRED}')
print()
print('C0-C3 la CPU thuan — KHONG can stage anh.')

Mounted at /content/drive
device: cuda
ensemble KHOA: ['EP0', 'R0_ep1', 'R0_ep2'] | rule mean_prob | bao cao TACH RIENG
cross-fit: 5-fold theo subject_id trong val
nguon prediction nb16: /content/drive/MyDrive/MIMIC_CXR/planB_out/label_reliability/preds

C0-C3 la CPU thuan — KHONG can stage anh.


## C1 · Nạp nhãn val + prediction đã lưu của nb16 — **không cần ảnh**

nb16 `C11` đã lưu `sid_val.npy`, `y_val.npy`, `subj_val.npy` nên phần này không phải dựng lại
cohort, không phải stage ảnh, không phải chạm BigQuery.

Prediction study-level của mọi `(arm, epoch)` cũng đã có sẵn — đó là toàn bộ nguyên liệu cho
`C3`.

In [4]:
SID_VAL  = np.load(f'{NB16_PRED}/sid_val.npy')
Y_VAL    = np.load(f'{NB16_PRED}/y_val.npy')
SUBJ_VAL = np.load(f'{NB16_PRED}/subj_val.npy')
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; C = len(LABELS); assert C == 13 and Y_VAL.shape[1] == C
cf = pickle.load(open(os.path.join(OUT_DIR, 'class_freq.pkl'), 'rb'))
N_POS_TRAIN = np.asarray(cf['class_freq'], 'float64'); N_TRAIN = int(cf['num_train'])
TIER = np.where(N_POS_TRAIN/N_TRAIN < 0.02, 'tail',
                np.where(N_POS_TRAIN/N_TRAIN < 0.10, 'medium', 'head'))
RARE3_IDX  = np.array([LABELS.index(l) for l in RARE3])
NONTGT_IDX = np.setdiff1d(np.arange(C), RARE3_IDX)
TAIL_IDX   = np.where(TIER == 'tail')[0]
W3_IDX     = np.array([LABELS.index(l) for l in WORST3_FIXED])
GRP = dict(macro=np.arange(C), rare3=RARE3_IDX, nontgt=NONTGT_IDX, tail=TAIL_IDX, worst3=W3_IDX)
n_st = len(SID_VAL)
print(f'VAL {n_st:,} study | {len(np.unique(SUBJ_VAL)):,} subject')
print(f'positives/nhan: ' + ' | '.join(f'{l} {int(Y_VAL[:, j].sum())}'
                                        for j, l in enumerate(LABELS) if l in RARE3))

# ---- kiem kho prediction cua nb16 ----
AVAIL = {}
if os.path.exists(f'{NB16_PRED}/EP0_s{SEED}.npy'): AVAIL['EP0'] = f'{NB16_PRED}/EP0_s{SEED}.npy'
for a in ('R0', 'R1GM', 'R2'):
    for e in range(1, 9):
        p = f'{NB16_PRED}/{a}_s{SEED}_ep{e}.npy'
        if os.path.exists(p): AVAIL[f'{a}_ep{e}'] = p
print()
print(f'prediction study-level co san: {len(AVAIL)}')
print(f'  {sorted(AVAIL)}')
_miss = [k for k in ENS_PRIMARY if k not in AVAIL]
assert not _miss, f'thieu prediction cho to hop ensemble da khoa: {_miss}'
P = {k: np.load(v) for k, v in AVAIL.items()}
for k, v in P.items():
    assert v.shape == (n_st, C), f'{k} shape {v.shape} != {(n_st, C)}'
    assert np.isfinite(v).all() and (v >= 0).all() and (v <= 1).all(), f'{k} khong phai xac suat'
print(f'  tat ca {len(P)} mang: shape dung, gia tri trong [0, 1]  OK')

VAL 31,670 study | 8,956 subject
positives/nhan: Fracture 690 | Lung_Lesion 1020 | Pleural_Other 377

prediction study-level co san: 25
  ['EP0', 'R0_ep1', 'R0_ep2', 'R0_ep3', 'R0_ep4', 'R0_ep5', 'R0_ep6', 'R0_ep7', 'R0_ep8', 'R1GM_ep1', 'R1GM_ep2', 'R1GM_ep3', 'R1GM_ep4', 'R1GM_ep5', 'R1GM_ep6', 'R1GM_ep7', 'R1GM_ep8', 'R2_ep1', 'R2_ep2', 'R2_ep3', 'R2_ep4', 'R2_ep5', 'R2_ep6', 'R2_ep7', 'R2_ep8']
  tat ca 25 mang: shape dung, gia tri trong [0, 1]  OK


## C2 · Bộ đo — AP nhanh · cross-fit theo subject · bootstrap

`ap_grouped` copy **verbatim** từ nb16 `C15`. Nó đã được kiểm khớp `sklearn` **bit-for-bit**
trên count bootstrap thật (`0/400` lệch, max `4.4e-16`), kể cả các ca biên: count `1..8`, điểm
làm tròn, mọi điểm bằng nhau, resample hết positive.

Lý do không dùng thẳng `sklearn`: mỗi resample phải `argsort` lại 31k phần tử ⇒ chậm hơn `11×`.

**Bẫy đã biết:** `k` bản sao của cùng một study **luôn** tie với nhau, nên bắt buộc gộp nhóm
điểm phân biệt rồi lấy precision ở **cuối** nhóm. Bản tính precision tại từng bản sao lệch tới
`2.2e-2` — lớn hơn mọi hiệu ứng đang xét cả một bậc. Cell dưới **kiểm lại** điều đó với count
bootstrap thật, không chỉ với `c = 1` (với `c = 1` thì không có tie nào và bản sai vẫn pass).

### Cross-fit

Chia val thành `K = 5` fold **theo `subject_id`**. Assert hai điều: fold rời nhau theo bệnh
nhân, và mỗi study xuất hiện **đúng một lần** trong tập dự đoán out-of-fold.

In [5]:
# ---------------- AP nhanh (VERBATIM nb16 C15) ----------------
def _prep(p):
    o = np.argsort(-p, kind='stable'); ps = p[o]
    gid = np.r_[0, np.cumsum(ps[1:] != ps[:-1])].astype('int64')
    return o, gid, int(gid[-1])+1

def ap_grouped(yo, co, gid, ng):
    a = np.bincount(gid, weights=yo*co, minlength=ng)
    n = np.bincount(gid, weights=co,    minlength=ng)
    npos = a.sum()
    if npos <= 0: return np.nan
    TP = np.cumsum(a); N = np.cumsum(n)
    m = a > 0
    return float((a[m]/npos*(TP[m]/N[m])).sum())

def make_prep(PS):
    pr = [_prep(PS[:, j]) for j in range(C)]
    return pr, [Y_VAL[pr[j][0], j].astype('float64') for j in range(C)]

def ap_of(pr, yo, c):
    out = np.full(C, np.nan)
    for j in range(C):
        o, gid, ng = pr[j]
        out[j] = ap_grouped(yo[j], c[o].astype('float64'), gid, ng)
    return out

# ---------------- GATE: phai khop sklearn ke ca khi CO tie ----------------
_ones = np.ones(n_st, 'int64')
_SU = np.unique(SUBJ_VAL); _six = np.searchsorted(_SU, SUBJ_VAL); _nsu = len(_SU)
_rg = np.random.default_rng(0); _bad = 0.0
_k0 = ENS_PRIMARY[0]; _pr0, _yo0 = make_prep(P[_k0])
for _t in range(3):
    _c = _ones if _t == 0 else np.bincount(_rg.integers(0, _nsu, _nsu), minlength=_nsu)[_six]
    _ix = np.repeat(np.arange(n_st), _c)
    _f = ap_of(_pr0, _yo0, _c)
    _r = np.array([average_precision_score(Y_VAL[_ix, j], P[_k0][_ix, j])
                   if Y_VAL[_ix, j].sum() else np.nan for j in range(C)])
    _bad = max(_bad, float(np.nanmax(np.abs(_f-_r))))
    print(f'  gate {_t+1}/3 [{"c=1, KHONG tie" if _t == 0 else "count bootstrap THAT, CO tie"}]'
          f'  max|diff| {_bad:.2e}')
assert _bad < 1e-9, f'AP tu cai lech sklearn {_bad:.3e} -> DUNG'
print('ap_grouped == sklearn ke ca khi co tie  OK')

# ---------------- cross-fit theo subject ----------------
def subject_folds(subj, K=K_FOLD, seed=SEED):
    su = np.unique(subj)
    rng = np.random.default_rng(seed)
    f_su = np.empty(len(su), 'int64'); f_su[rng.permutation(len(su))] = np.arange(len(su)) % K
    return pd.Series(subj).map(dict(zip(su, f_su))).to_numpy()
FOLD = subject_folds(SUBJ_VAL)
for k in range(K_FOLD):
    a = set(SUBJ_VAL[FOLD == k]); b = set(SUBJ_VAL[FOLD != k])
    assert a.isdisjoint(b), f'fold {k} chia se subject voi fold khac'
assert len(FOLD) == n_st and set(np.unique(FOLD)) == set(range(K_FOLD))
print()
print(f'cross-fit {K_FOLD} fold theo subject: '
      f'{[int((FOLD == k).sum()) for k in range(K_FOLD)]} study | '
      f'{[len(np.unique(SUBJ_VAL[FOLD == k])) for k in range(K_FOLD)]} subject')
print('  fold roi nhau theo subject: OK')

def cross_fit_predict(fit_fn, apply_fn):
    """fit_fn(idx_train) -> params ; apply_fn(params, idx_test) -> [n_test, C]. Tra ve OOF."""
    OOF = np.full((n_st, C), np.nan, 'float64'); seen = np.zeros(n_st, 'int64')
    for k in range(K_FOLD):
        tr = np.where(FOLD != k)[0]; te = np.where(FOLD == k)[0]
        OOF[te] = apply_fn(fit_fn(tr), te); seen[te] += 1
    assert (seen == 1).all(), 'co study bi du doan 0 hoac >1 lan trong OOF'
    assert np.isfinite(OOF).all(), 'OOF con NaN'
    return OOF

# ---------------- bootstrap (cluster = subject) ----------------
_rngb = np.random.default_rng(SEED)
BOOT_C = [np.bincount(_rngb.integers(0, _nsu, _nsu), minlength=_nsu)[_six] for _ in range(N_BOOT)]
print(f'bootstrap: {N_BOOT} resample cluster=subject_id da sinh (dung chung cho moi so sanh)')

def eval_full(PS, name=''):
    pr, yo = make_prep(PS)
    ap = ap_of(pr, yo, _ones)
    boot = {g: np.empty(N_BOOT) for g in GRP}
    for b, c in enumerate(BOOT_C):
        a = ap_of(pr, yo, c)
        for g, jj in GRP.items(): boot[g][b] = np.nanmean(a[jj])
    return dict(name=name, per_ap=ap,
                point={g: float(np.nanmean(ap[jj])) for g, jj in GRP.items()}, boot=boot)

def delta_ci(rA, rB, g):
    """A - B, paired tren cung resample."""
    d = rA['boot'][g] - rB['boot'][g]
    lo, hi = np.percentile(d, [2.5, 97.5])
    return (rA['point'][g]-rB['point'][g], float(lo), float(hi), bool(lo > 0 or hi < 0))
print('eval_full / delta_ci ready (moi so sanh dung CHUNG bo resample -> paired)')

  gate 1/3 [c=1, KHONG tie]  max|diff| 1.11e-16
  gate 2/3 [count bootstrap THAT, CO tie]  max|diff| 1.11e-16
  gate 3/3 [count bootstrap THAT, CO tie]  max|diff| 1.11e-16
ap_grouped == sklearn ke ca khi co tie  OK

cross-fit 5 fold theo subject: [6434, 6168, 6307, 6456, 6305] study | [1792, 1791, 1791, 1791, 1791] subject
  fold roi nhau theo subject: OK
bootstrap: 2000 resample cluster=subject_id da sinh (dung chung cho moi so sanh)
eval_full / delta_ci ready (moi so sanh dung CHUNG bo resample -> paired)


## C3 · **T1.3 — Ensemble** *(0 GPU, chạy được ngay)*

Trung bình xác suất của tổ hợp đã khoá `{ep0, R0 ep1, R0 ep2}`.

Không cần cross-fit: trung bình đơn thuần **không có tham số nào fit từ dữ liệu**. Cái cần
kiểm soát là việc *chọn* tổ hợp — và tổ hợp đã khoá ở `C0` theo một tiêu chí không tham chiếu
hiệu năng (ba điểm trong dung sai macro `0.0015` của rule §9).

Bảng thứ hai liệt kê các tổ hợp/quy tắc khác. Chúng là **exploratory** — chọn số tốt nhất
trong bảng đó rồi báo cáo là chính xác cái lỗi mà ràng buộc phương pháp ở đầu notebook cấm.

### Ba quy tắc gộp

| | công thức | ghi chú |
|---|---|---|
| `mean_prob` | `mean(p_i)` | **primary**, đơn giản nhất |
| `mean_logit` | `σ(mean(logit p_i))` | nhạy hơn ở đuôi |
| `mean_rank` | trung bình thứ hạng, chuẩn hoá | AP chỉ phụ thuộc thứ hạng nên đây thường tốt cho AP |

`mean_rank` **không** thay đổi AP nếu chỉ có một model — nó chỉ có nghĩa khi gộp nhiều model
có thang điểm khác nhau.

In [6]:
def ens_mean_prob(keys):  return np.mean([P[k] for k in keys], 0)
def ens_mean_logit(keys):
    lg = [np.log(np.clip(P[k], 1e-7, 1-1e-7)/(1-np.clip(P[k], 1e-7, 1-1e-7))) for k in keys]
    z = np.mean(lg, 0); return 1/(1+np.exp(-z))
def ens_mean_rank(keys):
    R = np.zeros((n_st, C))
    for k in keys:
        for j in range(C):
            R[:, j] += pd.Series(P[k][:, j]).rank(method='average').to_numpy()
    return R/(len(keys)*n_st)
ENS_FN = dict(mean_prob=ens_mean_prob, mean_logit=ens_mean_logit, mean_rank=ens_mean_rank)

BASE_KEY = REF_MODEL
R_BASE = eval_full(P[BASE_KEY], f'single {BASE_KEY}')
R_EP0  = eval_full(P['EP0'], 'single EP0 (chua train)')
R_ENS  = eval_full(ENS_FN[ENS_RULE](ENS_PRIMARY), f'ENSEMBLE {ENS_RULE} {ENS_PRIMARY}')

print('=== SINGLE-MODEL (con so CHINH) ===')
for r in (R_EP0, R_BASE):
    print(f'  {r["name"]:<30} ' + ' | '.join(f'{g} {r["point"][g]:.4f}' for g in GRP))
print()
print('=== ENSEMBLE (dong RIENG, khong tron vao con so chinh) ===')
print(f'  {R_ENS["name"]:<30} ' + ' | '.join(f'{g} {R_ENS["point"][g]:.4f}' for g in GRP))
print()
print(f'=== ENSEMBLE - single({BASE_KEY})   [paired, cluster=subject, {N_BOOT} resample] ===')
rows = []
for g in GRP:
    d, lo, hi, ex = delta_ci(R_ENS, R_BASE, g)
    rows.append(dict(metric=g, delta=d, ci_lo=lo, ci_hi=hi, excl0=ex))
    print(f'  {g:<8} {d:+.5f}  CI95 [{lo:+.5f}, {hi:+.5f}]  {"LOAI 0" if ex else "chua 0"}')
T_ENS = pd.DataFrame(rows)

print()
print('=== EXPLORATORY: to hop / quy tac khac (KHONG duoc chon so tot nhat o day de bao cao) ===')
_combos = {'primary(khoa)': ENS_PRIMARY,
           'R0 ep1+ep2': ['R0_ep1', 'R0_ep2'],
           'EP0+R0ep2': ['EP0', 'R0_ep2'],
           'R0 ep1..3': [k for k in ['R0_ep1', 'R0_ep2', 'R0_ep3'] if k in P],
           'R0 tat ca ep': [k for k in P if k.startswith('R0_ep')],
           '3 arm @ep2': [k for k in ['R0_ep2', 'R1GM_ep2', 'R2_ep2'] if k in P]}
rows = []
for nm, ks in _combos.items():
    if any(k not in P for k in ks) or not ks: continue
    for rule, fn in ENS_FN.items():
        ap = ap_of(*make_prep(fn(ks)), _ones)
        rows.append(dict(combo=nm, k=len(ks), rule=rule,
                         **{g: float(np.nanmean(ap[jj])) for g, jj in GRP.items()}))
T_EXP = pd.DataFrame(rows)
print(T_EXP.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
T_EXP.to_csv(f'{SC_DIR}/metrics/ensemble_exploratory.csv', index=False)

_ens_json = dict(primary=dict(combo=ENS_PRIMARY, rule=ENS_RULE, point=R_ENS['point']),
                 single_ref=dict(name=BASE_KEY, point=R_BASE['point']),
                 single_ep0=dict(point=R_EP0['point']),
                 delta_vs_single=T_ENS.to_dict('records'),
                 exploratory=T_EXP.to_dict('records'))
json.dump(_ens_json, open(f'{SC_DIR}/metrics/ensemble.json', 'w'), indent=1)
print()
print(f'-> {SC_DIR}/metrics/ensemble.json')
print()
print('  NHAC: ensemble la mot dong RIENG trong bao cao. T2/T3 nham vao con so SINGLE-MODEL,')
print('  khong bi doi phai thang ensemble — hai thu tra loi hai cau hoi khac nhau.')

=== SINGLE-MODEL (con so CHINH) ===
  single EP0 (chua train)        macro 0.4355 | rare3 0.1478 | nontgt 0.5218 | tail 0.1218 | worst3 0.1360
  single R0_ep2                  macro 0.4365 | rare3 0.1467 | nontgt 0.5234 | tail 0.1191 | worst3 0.1338

=== ENSEMBLE (dong RIENG, khong tron vao con so chinh) ===
  ENSEMBLE mean_prob ['EP0', 'R0_ep1', 'R0_ep2'] macro 0.4376 | rare3 0.1496 | nontgt 0.5240 | tail 0.1238 | worst3 0.1373

=== ENSEMBLE - single(R0_ep2)   [paired, cluster=subject, 2000 resample] ===
  macro    +0.00112  CI95 [+0.00027, +0.00197]  LOAI 0
  rare3    +0.00291  CI95 [-0.00022, +0.00610]  chua 0
  nontgt   +0.00058  CI95 [+0.00008, +0.00108]  LOAI 0
  tail     +0.00465  CI95 [-0.00230, +0.01213]  chua 0
  worst3   +0.00345  CI95 [+0.00035, +0.00645]  LOAI 0

=== EXPLORATORY: to hop / quy tac khac (KHONG duoc chon so tot nhat o day de bao cao) ===
        combo  k       rule  macro  rare3  nontgt   tail  worst3
primary(khoa)  3  mean_prob 0.4376 0.1496  0.5240 0.1238  

## C4 · Stage ảnh + dataset + model *(bắt đầu phần GPU)*

Từ đây mới cần ảnh. `C0b` staging lấy verbatim từ nb13/nb16 — có cờ `.done_<shard>` nên chạy
lại là tiếp tục. **Lần đầu ~40–70 phút**, IO thuần, Drive trần ~83 MB/s.

Manifest ảnh dựng **y hệt nb16 `C1`** (cùng bộ lọc, cùng thứ tự sort) — bắt buộc, vì `C5` sẽ
assert rằng pool lại per-image phải tái tạo **bit-for-bit** file study-level nb16 đã lưu. Lệch
một ảnh là assert đó fail.

Dataset ở đây khác nb16 một điểm: trả về **ảnh 560 nguyên**, chưa crop. Mọi biến thể transform
được dẫn xuất trên GPU từ cùng một tensor, nên chỉ đọc đĩa **một lần** cho cả ba biến thể.

In [9]:
# ============ STAGE CHI SHARD CHUA ANH VAL ============
# nb17 chi doc anh VAL (C5 chay raw_loader(IDX_VAL)). Stage ca 346k anh nhu nb16 la thua ~6x.
# 04a sort theo (subject, study, dicom) roi cat shard, va split gan o muc SUBJECT,
# nen anh val don thanh mot dai shard gan lien nhau -> loc duoc bang _work_list.parquet.
os.makedirs(LOCAL_IMG, exist_ok=True)
_WL = os.path.join(SHARD_DIR, '_work_list.parquet')
_have0 = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
print(f'da co {_have0:,} anh o {LOCAL_IMG}')

if os.path.isdir(SHARD_DIR) and os.path.exists(_WL):
    wl = pd.read_parquet(_WL)
    need = sorted(wl.loc[wl.split == 'val', 'shard'].unique().tolist())
    n_val_img = int((wl.split == 'val').sum())
    print(f'shard tong {wl.shard.nunique()} | shard CHUA anh val: {len(need)} '
          f'({len(need)/wl.shard.nunique():.0%}) | anh val {n_val_img:,}')
elif os.path.isdir(SHARD_DIR):
    need = [int(f[6:11]) for f in sorted(os.listdir(SHARD_DIR)) if f.endswith('.tar')]
    print(f'khong thay _work_list.parquet -> stage TAT CA {len(need)} shard')
else:
    need = []
    assert _have0 > 1000, f'khong co shard o {SHARD_DIR} va {LOCAL_IMG} rong'
    print('khong co SHARD_DIR -> dung anh da co san')

todo = [i for i in need if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_shard_{i:05d}.tar'))]
print(f'can stage: {len(todo)} shard')
if todo:
    t0, mb = time.time(), 0.0
    bar = tqdm(todo, desc='stage shard val', unit='shard')
    for i in bar:
        f = f'shard_{i:05d}.tar'; src = os.path.join(SHARD_DIR, f); tmp = f'/content/{f}'
        mb += os.path.getsize(src)/1e6
        shutil.copyfile(src, tmp)
        with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
        os.remove(tmp); open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()
        el = time.time()-t0
        bar.set_postfix_str(f'{mb/max(el,1e-9):.0f} MB/s | {mb:.0f} MB | con ~'
                            f'{el/max(bar.n,1)*(len(todo)-bar.n)/60:.0f} ph')
    print(f'xong {mb:.0f} MB trong {(time.time()-t0)/60:.1f} phut ({mb/(time.time()-t0):.0f} MB/s)')
print('local images:', len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')]))
print()
print('  Ngat giua chung KHONG sao: co co .done_<shard> nen chay lai la tiep tuc.')
print('  Neu MB/s tut xuong <10 thi Drive dang bi throttle -> doi vai phut roi chay lai.')

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T
from PIL import Image
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
assert proc.crop_size['height'] == MODEL_SIZE
MEAN, STD = proc.image_mean[0], proc.image_std[0]
raw_tf = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])     # KHONG crop

# ---------------- DO STORE_SIZE tu anh THAT, khong tin ten thu muc ----------------
_one = next(f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg'))
with Image.open(os.path.join(LOCAL_IMG, _one)) as _im: _w, _h = _im.size
assert _w == _h, f'anh luu khong vuong: {_w}x{_h}'
STORE_SIZE = _w
_expect = int(round(MODEL_SIZE*1.08))
print(f'STORE_SIZE do tu {_one}: {STORE_SIZE}px  (04a: round(518*1.08) = {_expect})')
assert STORE_SIZE >= MODEL_SIZE, f'anh luu {STORE_SIZE} < model {MODEL_SIZE}'
if STORE_SIZE != 560:
    print(f"  ! ten thu muc la 'cxr_560_shards' nhung kich thuoc THAT la {STORE_SIZE}.")
    print(f"    Moi tinh toan crop/FOV phai dung {STORE_SIZE}.")

# ---------------- manifest: khop nb16 C1, nhung chi doi anh VAL ton tai ----------------
# nb16 loc ton-tai tren TOAN BO man. O day chi stage shard val nen train/test se vang —
# khong sao: loc theo dicom_id da sort thi viec bo dong train KHONG doi thu tu tuong doi
# cua cac dong val, nen man.iloc[IDX_VAL] van trung nb16.
ehr = (pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet'),
                       columns=['study_id', 'subject_id', 'split'])
       .sort_values('study_id').reset_index(drop=True))
vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal', 'lateral'])]
_dup = [c for c in ('subject_id', 'split', 'setting') if c in vm.columns]
man = (vm.drop(columns=_dup).merge(ehr[['study_id', 'subject_id', 'split']], on='study_id', how='inner')
       .sort_values('dicom_id').reset_index(drop=True))
_exists = man.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))
_isval = man.split.to_numpy() == 'val'
_miss_val = int((_isval & ~_exists.to_numpy()).sum())
print(f'anh val trong manifest {int(_isval.sum()):,} | thieu tren dia {_miss_val:,} '
      f'({_miss_val/max(int(_isval.sum()), 1):.2%})')
assert _miss_val/max(int(_isval.sum()), 1) < 0.01,     f'{_miss_val} anh VAL thieu -> stage chua xong hoac shard loi'
man = man[_exists.to_numpy() | ~_isval].reset_index(drop=True)   # giu train/test du vang
IDX_VAL = np.where(man.split.to_numpy() == 'val')[0]
_sid_chk = np.sort(man.iloc[IDX_VAL].study_id.unique())
assert np.array_equal(_sid_chk, SID_VAL), \
    f'thu tu study val KHAC nb16 ({len(_sid_chk)} vs {len(SID_VAL)}) -> manifest khong khop'
print(f'anh val {len(IDX_VAL):,} | study val {len(SID_VAL):,}  (khop nb16)')
_vg = man.iloc[IDX_VAL].view_group.value_counts()
print(f'  theo view: {dict(_vg)}')
_per = man.iloc[IDX_VAL].groupby('study_id').view_group.agg(lambda s: ''.join(sorted(set(x[0].upper() for x in s))))
print(f'  chu ky view cua study: {dict(_per.value_counts())}')
_both = int((_per == 'FL').sum())
print(f'  study co CA HAI view: {_both:,} ({_both/len(SID_VAL):.1%})  <- W_F chi anh huong nhom nay')

class RawDS(Dataset):
    """Tra ve anh STORE_SIZE NGUYEN; moi crop deu duoc dan xuat tren GPU."""
    def __init__(s, idx): s.idx = np.asarray(idx)
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        r = man.iloc[s.idx[k]]
        im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
        return raw_tf(Image.merge('RGB', (im, im, im)))
raw_loader = lambda idx, bs, w=4: DataLoader(RawDS(idx), batch_size=bs, shuffle=False,
                                             num_workers=w, pin_memory=(dev == 'cuda'),
                                             persistent_workers=False)

# ---------------- model: VERBATIM nb16 C6 ----------------
TRAIN_FROM = 9
_sd04b = torch.load(BB_CKPT, map_location='cpu', weights_only=False)['model']
BB_SD = {k[9:]: v for k, v in _sd04b.items() if k.startswith('backbone.')}
def load_backbone(device=dev):
    m = AutoModel.from_pretrained('microsoft/rad-dino'); m.load_state_dict(BB_SD, strict=True)
    return m.to(device).eval()
backbone = load_backbone()
for p in backbone.parameters(): p.requires_grad_(False)
LAYERS = backbone.encoder.layer; N_BLOCKS = len(LAYERS); D = backbone.config.hidden_size
N_PATCH = (MODEL_SIZE//14)**2
def _layer(h, i):
    o = LAYERS[i](h); return o[0] if isinstance(o, (tuple, list)) else o
@torch.no_grad()
def forward_shared(x):
    h = backbone.embeddings(x)
    for i in range(TRAIN_FROM): h = _layer(h, i)
    return h.detach()

import sys, types
ML_SRC = '/content/ml_decoder.py'
if not os.path.exists(ML_SRC):
    !wget -q -O {ML_SRC} https://raw.githubusercontent.com/Alibaba-MIIL/ML_Decoder/main/src_files/ml_decoder/ml_decoder.py
sys.path.insert(0, '/content')
import ml_decoder as _mld
from ml_decoder import MLDecoder
if not hasattr(_mld.TransformerDecoderLayerOptimal, '_patched'):
    _oi = _mld.TransformerDecoderLayerOptimal.__init__
    def _init(self, *a, **kw): _oi(self, *a, **kw); self.self_attn = types.SimpleNamespace(batch_first=False)
    _mld.TransformerDecoderLayerOptimal.__init__ = _init
    _of = _mld.TransformerDecoderLayerOptimal.forward
    def _fwd(self, tgt, memory, **kw):
        kw.pop('tgt_is_causal', None); kw.pop('memory_is_causal', None); return _of(self, tgt, memory, **kw)
    _mld.TransformerDecoderLayerOptimal.forward = _fwd
    _mld.TransformerDecoderLayerOptimal._patched = True
H5_SD = torch.load(H5_CKPT, map_location='cpu', weights_only=False)['model']
_sub = lambda p: {k[len(p):]: v for k, v in H5_SD.items() if k.startswith(p)}
_TAIL_SRC = load_backbone('cpu')
for p in _TAIL_SRC.parameters(): p.requires_grad_(False)

class TailBackbone(nn.Module):
    def __init__(s):
        super().__init__()
        s.blocks = nn.ModuleList([copy.deepcopy(_TAIL_SRC.encoder.layer[i])
                                  for i in range(TRAIN_FROM, N_BLOCKS)])
        s.ln = copy.deepcopy(_TAIL_SRC.layernorm)
    def forward(s, h):
        for b in s.blocks:
            o = b(h); h = o[0] if isinstance(o, (tuple, list)) else o
        return s.ln(h)
class H5Head(nn.Module):
    def __init__(s):
        super().__init__()
        s.norm = nn.LayerNorm(D)
        s.mld = MLDecoder(num_classes=C, initial_num_features=D, decoder_embedding=D, num_of_groups=C)
        s.norm.load_state_dict(_sub('norm.')); s.mld.load_state_dict(_sub('mld.'))
    def forward(s, patches): return s.mld(s.norm(patches))
class Arm(nn.Module):
    def __init__(s):
        super().__init__(); s.tail = TailBackbone(); s.head = H5Head()
    def forward(s, z):
        h = s.tail(z)
        assert h.shape[1] == N_PATCH+1
        return s.head(h[:, 1:])                     # PATCH-ONLY

def load_ref(name=REF_MODEL):
    p = f'{NB16_SNAP}/{name.replace("_ep", f"_s{SEED}_ep")}.pt'
    assert os.path.exists(p), f'khong thay snapshot {p}'
    m = Arm().to(dev); m.load_state_dict({k: v.to(dev) for k, v in
                                          torch.load(p, map_location='cpu', weights_only=False).items()})
    return m.eval()
MODEL = load_ref(REF_MODEL)
print()
print(f'model tham chieu: {REF_MODEL} <- snapshot nb16 | patch-only, {N_PATCH} token')

da co 346,578 anh o /content/cxr_local
shard tong 347 | shard CHUA anh val: 347 (100%) | anh val 51,880
can stage: 0 shard
local images: 346578

  Ngat giua chung KHONG sao: co co .done_<shard> nen chay lai la tiep tuc.
  Neu MB/s tut xuong <10 thi Drive dang bi throttle -> doi vai phut roi chay lai.
STORE_SIZE do tu 471eb574-ae384849-1c525269-a52023de-4a6b2f04.jpg: 559px  (04a: round(518*1.08) = 559)
  ! ten thu muc la 'cxr_560_shards' nhung kich thuoc THAT la 559.
    Moi tinh toan crop/FOV phai dung 559.
anh val trong manifest 51,880 | thieu tren dia 0 (0.00%)
anh val 51,880 | study val 31,670  (khop nb16)
  theo view: {'frontal': np.int64(35061), 'lateral': np.int64(16819)}
  chu ky view cua study: {'F': np.int64(16314), 'FL': np.int64(15299), 'L': np.int64(57)}
  study co CA HAI view: 15,299 (48.3%)  <- W_F chi anh huong nhom nay


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]


model tham chieu: R0_ep2 <- snapshot nb16 | patch-only, 1369 token


## C5 · **T1.1 + T1.2 / F2** — per-image + ablation transform

Một lượt đọc đĩa, ba biến thể dẫn xuất trên GPU từ cùng tensor 560:

| biến thể | cách lấy | FOV |
|---|---|---|
| `center518` | `x[:, :, 21:539, 21:539]` | **hiện hành** — vứt viền `21px` mỗi bên |
| `resize518` | `F.interpolate(560→518, bicubic, antialias)` | **giữ trọn** khung 560, độ phân giải thấp hơn chút |
| `tta5crop` | 4 góc + tâm, trung bình xác suất | phủ hết khung 560 |

Ảnh lưu ở `04a` **đã là 560×560 vuông**, nên `resize518` là thu nhỏ đều — **không** méo tỉ lệ.
So sánh `center518` vs `resize518` vì thế sạch: khác nhau đúng ở phần FOV bị vứt.

### ⚠ Đây chỉ là **lần cắt thứ hai**

`04a_preprocess_images.ipynb` cell 8 đã **cắt lần một** trước đó: shortest-edge resize rồi
center-crop `560×560`. Phim ngực dạng đứng nên lần cắt đó bỏ **đỉnh phổi** và **góc sườn
hoành** — lớn hơn nhiều so với viền `21px` ở đây.

**`C5` âm không bác bỏ giả thuyết nút thắt ngoại vi.** Chỉ `19_fov_ablation.ipynb` (`F1`/`F3`)
mới chạm tới lần cắt một. Ghi trước để một kết quả âm không đóng nhầm câu hỏi.

### Cổng kiểm bắt buộc

Pool lại per-image `center518` bằng `W_F = 0.7 / W_L = 0.3` phải tái tạo **bit-for-bit** file
`R0_s42_ep2.npy` mà nb16 đã lưu. Nếu không khớp thì đường per-image sai ở đâu đó, và mọi kết
luận multi-view ở `C6` đều vô nghĩa.

**Chi phí:** 6 lượt forward (1 center + 1 resize + 4 góc) trên `51,880` ảnh val ≈ **35–45 phút**
GPU. Cache ra `.npy`, chạy lại là đọc cache.

In [13]:
PI_NPY = lambda t: f'{SC_DIR}/preds/perimg_val_{REF_MODEL}_{t}.npy'
OFF = STORE_SIZE - MODEL_SIZE
# torchvision center_crop dung int(round(OFF/2)), KHONG phai OFF//2. Hai cai lech nhau
# khi OFF le va OFF//2 la so le (vd OFF=43: 21 vs 22). Phai dung dung quy tac cua no,
# neu khong crop 'center' cua ta lech 1px so voi eval_tf cua nb16.
C_OFF = int(round(OFF/2.0))
CROPS = dict(center=(C_OFF, C_OFF), tl=(0, 0), tr=(0, OFF), bl=(OFF, 0), br=(OFF, OFF))
BS_EVAL = 64
print(f'STORE {STORE_SIZE} -> MODEL {MODEL_SIZE} | OFF {OFF} | center offset {C_OFF}')

# ---- GATE: lat cat 'center' phai TRUNG torchvision center_crop, bit-for-bit ----
_t = torch.randn(2, 3, STORE_SIZE, STORE_SIZE)
_mine = _t[:, :, C_OFF:C_OFF+MODEL_SIZE, C_OFF:C_OFF+MODEL_SIZE]
_tv = T.functional.center_crop(_t, [MODEL_SIZE, MODEL_SIZE])
assert torch.equal(_mine, _tv), ('lat cat center KHAC torchvision center_crop -> se lech '
                                 'voi eval_tf cua nb16 va cong tai-tao se fail')
print(f'  gate: slice[{C_OFF}:{C_OFF+MODEL_SIZE}] == T.functional.center_crop  bit-for-bit  OK')

TTA = False           # <<< True de them 4 goc (tta5crop). Xem ghi chu chi phi ben duoi.
VARIANTS = ['center', 'resize'] + (['tl', 'tr', 'bl', 'br'] if TTA else [])
TRANSFORMS = ['center518', 'resize518'] + (['tta5crop'] if TTA else [])

@torch.no_grad()
def _build(xb):
    """Dung TAT CA bien the thanh MOT tensor -> mot forward, mot sync."""
    out = []
    for nm in VARIANTS:
        if nm == 'resize':
            out.append(F.interpolate(xb, size=(MODEL_SIZE, MODEL_SIZE), mode='bicubic',
                                     align_corners=False, antialias=True))
        else:
            i0, j0 = CROPS[nm]
            out.append(xb[:, :, i0:i0+MODEL_SIZE, j0:j0+MODEL_SIZE])
    return torch.cat(out, 0)

@torch.no_grad()
def _probs_all(xb, chunk=None):
    """[B,3,S,S] -> {bien the: [B, C]}. Mot lan .cpu() cho ca lo."""
    big = _build(xb); k = xb.shape[0]
    if chunk is None: chunk = big.shape[0]
    ps = []
    for i in range(0, big.shape[0], chunk):
        ps.append(torch.sigmoid(MODEL(forward_shared(big[i:i+chunk]))))
    p = torch.cat(ps, 0).float().cpu().numpy()
    return {nm: p[t*k:(t+1)*k] for t, nm in enumerate(VARIANTS)}

if all(os.path.exists(PI_NPY(t)) for t in TRANSFORMS):
    PIMG = {t: np.load(PI_NPY(t)) for t in TRANSFORMS}
    print('per-image <- cache')
else:
    acc = {nm: np.zeros((len(IDX_VAL), C), 'float32') for nm in VARIANTS}
    pos, t0, chunk = 0, time.time(), None
    bar = tqdm(raw_loader(IDX_VAL, BS_EVAL), desc=f'per-image val ({len(VARIANTS)} bien the)')
    for xb in bar:
        xb = xb.to(dev, non_blocking=True); k = xb.shape[0]
        assert xb.shape[-1] == STORE_SIZE, f'anh khong phai {STORE_SIZE}px: {tuple(xb.shape)}'
        while True:
            try:
                got = _probs_all(xb, chunk); break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                chunk = max(k, (chunk or k*len(VARIANTS))//2)
                print(f'  OOM -> ha chunk xuong {chunk}')
                if chunk <= k: chunk = k
        for nm in VARIANTS: acc[nm][pos:pos+k] = got[nm]
        pos += k
        if bar.n and bar.n % 25 == 0:
            _el = time.time()-t0
            bar.set_postfix_str(f'{pos/_el:.0f} img/s | con ~{_el/bar.n*(len(bar)-bar.n)/60:.0f} ph')
    assert pos == len(IDX_VAL)
    PIMG = {'center518': acc['center'], 'resize518': acc['resize']}
    if TTA: PIMG['tta5crop'] = np.mean([acc[n] for n in ('center', 'tl', 'tr', 'bl', 'br')], 0)
    for t, v in PIMG.items(): np.save(PI_NPY(t), v)
    print(f'xong sau {(time.time()-t0)/60:.1f} phut -> {SC_DIR}/preds/')

# ---------------- pooling (VERBATIM nb16 pool_study, viet lai theo per-image) ----------------
_sub_man = man.iloc[IDX_VAL]
VG   = _sub_man.view_group.to_numpy()
SIDI = pd.Series(_sub_man.study_id.to_numpy()).map({s: i for i, s in enumerate(SID_VAL)}).to_numpy()
IS_F, IS_L = VG == 'frontal', VG == 'lateral'
def _agg(pi, mask, how='mean'):
    """Gop anh trong mot view -> [n_study, C]; NaN neu study khong co view do."""
    n = np.bincount(SIDI[mask], minlength=len(SID_VAL)).astype('float64')
    out = np.full((len(SID_VAL), C), np.nan)
    for j in range(C):
        v = pi[mask, j].astype('float64')
        if how == 'mean':
            s = np.bincount(SIDI[mask], weights=v, minlength=len(SID_VAL))
            out[n > 0, j] = (s/np.maximum(n, 1))[n > 0]
        elif how == 'max':
            m = np.full(len(SID_VAL), -np.inf); np.maximum.at(m, SIDI[mask], v)
            out[n > 0, j] = m[n > 0]
        elif how == 'lse':
            # log-mean-exp = logsumexp - log(n): noi suy mean<->max VA bat bien theo so anh.
            # logsumexp tho se thoi diem cua study nhieu anh len chi vi no nhieu anh.
            m = np.full(len(SID_VAL), -np.inf); np.maximum.at(m, SIDI[mask], v)
            e = np.bincount(SIDI[mask], weights=np.exp(v-m[SIDI[mask]]), minlength=len(SID_VAL))
            out[n > 0, j] = (m + np.log(np.maximum(e, 1e-300)) - np.log(np.maximum(n, 1)))[n > 0]
    return out, (n > 0)
def pool(pi, wf=W_F, wl=W_L, how='mean'):
    fr, hf = _agg(pi, IS_F, how); la, hl = _agg(pi, IS_L, how)
    wf = np.broadcast_to(np.asarray(wf, 'float64').reshape(1, -1), (len(SID_VAL), C))
    wl = np.broadcast_to(np.asarray(wl, 'float64').reshape(1, -1), (len(SID_VAL), C))
    num = wf*np.nan_to_num(fr)*hf[:, None] + wl*np.nan_to_num(la)*hl[:, None]
    den = wf*hf[:, None] + wl*hl[:, None]
    return (num/np.maximum(den, 1e-30)).astype('float32')

# ---------------- COÅNG: tai tao ket qua cua nb16 ----------------
# Bai hoc tu lan chay dau: nguong tren LECH TUYET DOI la sai cong cu.
#   trong CUNG mot session, hai duong tinh trung nhau ~1e-7
#   qua session/GPU khac, kernel cuBLAS khac cho ~1e-3 tren logit -> ~1e-4 tren xac suat
# Nguong 1e-4 hieu chuan o che do thu nhat roi ap cho che do thu hai => bao dong gia.
# Cong THAT phai ben voi nhieu so hoc nhung vo khi sai cau truc:
#   A. Spearman tung nhan  (AP chi phu thuoc THU HANG; jitter 1e-4 khong xao thu hang)
#   B. AP khop dung so nb16 DA LOG  (do thang thu duoc dung)
_re = pool(PIMG['center518'])
_ad = np.abs(_re.astype('float64') - P[REF_MODEL].astype('float64'))
_d = float(_ad.max())

print()
print('=== MOI TRUONG: nb16 vs bay gio ===')
_gpu_now = torch.cuda.get_device_name(0) if dev == 'cuda' else 'cpu'
try:
    _lk = json.load(open(f'{NB16_DIR}/locked_config.json'))
    _gpu16 = _lk['env']['gpu']; _tor16 = _lk['env']['torch']
except Exception:
    _gpu16 = _tor16 = '(khong doc duoc locked_config.json)'
print(f'  GPU   nb16 {_gpu16}  |  bay gio {_gpu_now}')
print(f'  torch nb16 {_tor16}  |  bay gio {torch.__version__}')
_same_env = (_gpu16 == _gpu_now) and (_tor16 == torch.__version__)
print(f'  -> moi truong {"TRUNG" if _same_env else "KHAC"}'
      + ('' if _same_env else '  => lech ~1e-4 tren xac suat la BINH THUONG'))

print()
print(f'=== DIAGNOSTIC (KHONG phai cong): |pool(per-image) - {REF_MODEL}.npy| ===')
print(f'  max {_d:.3e} | trung vi {float(np.median(_ad)):.3e} | p99 {float(np.percentile(_ad, 99)):.3e}')

def _spearman(u, v):
    ru = pd.Series(u).rank().to_numpy(); rv = pd.Series(v).rank().to_numpy()
    ru = ru - ru.mean(); rv = rv - rv.mean()
    return float((ru*rv).sum()/max(np.sqrt((ru**2).sum()*(rv**2).sum()), 1e-30))
_rho = np.array([_spearman(_re[:, j], P[REF_MODEL][:, j]) for j in range(C)])
print()
print('=== COÅNG A - Spearman tung nhan (>= 0.999) ===')
print('  ' + ' | '.join(f'{LABELS[j][:9]} {_rho[j]:.5f}' for j in range(C)))
print(f'  thap nhat: {LABELS[int(_rho.argmin())]} {_rho.min():.6f}  '
      f'{"PASS" if _rho.min() >= 0.999 else "FAIL"}')

_arm, _epn = REF_MODEL.split('_ep')
try:
    _log16 = json.load(open(f'{NB16_DIR}/metrics/epoch_log_seed{SEED}.json'))
    _ap16 = np.array(_log16[_arm][_epn]['per_ap'])
    _m16 = {g: float(np.nanmean(_ap16[jj])) for g, jj in GRP.items()}
except Exception as _e:
    _m16 = None; print(f'  ! khong doc duoc epoch_log cua nb16: {_e}')
_apme = ap_of(*make_prep(_re), _ones)
_mme = {g: float(np.nanmean(_apme[jj])) for g, jj in GRP.items()}
print()
print('=== COÅNG B - AP tai tao vs AP nb16 DA LOG (<= 2e-4) ===')
_gapmax = 0.0
if _m16:
    for g in GRP:
        _gap = _mme[g]-_m16[g]; _gapmax = max(_gapmax, abs(_gap))
        print(f'  {g:<8} tai tao {_mme[g]:.6f} | nb16 {_m16[g]:.6f} | lech {_gap:+.6f}'
              + ('  <-- vuot 2e-4' if abs(_gap) > 2e-4 else ''))
    print(f'  lech lon nhat {_gapmax:.2e}  {"PASS" if _gapmax <= 2e-4 else "FAIL"}')

_okA = bool(_rho.min() >= 0.999)
_okB = bool(_m16 is None or _gapmax <= 2e-4)
if not (_okA and _okB):
    print()
    print('  ===== DINH VI LECH =====')
    print('  [1] phan bo lech theo STUDY')
    _ps = _ad.max(1)
    for th in (1e-6, 1e-4, 1e-3, 1e-2, 1e-1):
        print(f'      study co max|diff| > {th:.0e}: {int((_ps > th).sum()):>7,} / {len(_ps):,} '
              f'({(_ps > th).mean():.2%})')
    print('  [2] lech theo CHU KY VIEW (F-only / L-only / FL)')
    _sig = pd.Series(_per.reindex(SID_VAL).fillna('').to_numpy())
    for k, m in _sig.groupby(_sig).groups.items():
        mi = np.asarray(list(m))
        print(f'      {k or "(none)":<8} n={len(mi):>6,}  max {_ps[mi].max():.3e}  '
              f'trung vi {np.median(_ps[mi]):.3e}')
    print('      -> chi FL lech => loi TRONG SO/gop view. Moi chu ky deu lech => ANH/MODEL/so hoc.')
    print('  [3] lech theo NHAN')
    print('      ' + ' | '.join(f'{LABELS[j][:11]} {_ad.max(0)[j]:.2e}' for j in range(C)))
    print('  [4] DUONG DAN nb16 NGUYEN BAN vs duong cua ta, tren 256 anh dau')
    print('      LUU Y: kiem nay dung CUNG mot MODEL cho ca hai nhanh nen KHONG the phat')
    print('      hien lech MODEL. No chi kiem TRANSFORM. Cong B moi la cai kiem model.')
    _n = min(256, len(IDX_VAL))
    class _NB16DS(torch.utils.data.Dataset):
        def __init__(s, idx): s.idx = np.asarray(idx)
        def __len__(s): return len(s.idx)
        def __getitem__(s, k):
            r = man.iloc[s.idx[k]]
            im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
            im = Image.merge('RGB', (im, im, im))
            return T.Compose([T.CenterCrop(MODEL_SIZE), T.ToTensor(),
                              T.Normalize([MEAN]*3, [STD]*3)])(im)
    _p16 = []
    with torch.no_grad():
        for _x in torch.utils.data.DataLoader(_NB16DS(IDX_VAL[:_n]), batch_size=32, num_workers=2):
            _p16.append(torch.sigmoid(MODEL(forward_shared(_x.to(dev)))).float().cpu().numpy())
    _dp = float(np.abs(np.concatenate(_p16, 0) - PIMG['center518'][:_n]).max())
    print(f'      max|diff| {_dp:.3e} -> '
          + ('LECH O TRANSFORM' if _dp > 1e-5 else 'transform TRUNG'))
    print('  [5] so anh moi study (tap anh khac nb16 se lam mean trong view lech)')
    _cnt = pd.Series(SIDI).value_counts().sort_index()
    print(f'      anh/study: min {_cnt.min()} | trung vi {_cnt.median():.0f} | max {_cnt.max()} '
          f'| tong {int(_cnt.sum()):,}')
    print('  =========================')
    print()
    raise AssertionError(
        f'tai tao KHONG dat cong cau truc (Spearman min {_rho.min():.6f}, '
        f'lech AP {_gapmax:.2e}). Doc khoi DINH VI o tren.')

print()
print(f'  COÅNG A Spearman min {_rho.min():.6f} >= 0.999   PASS')
if _m16: print(f'  COÅNG B lech AP {_gapmax:.2e} <= 2e-4        PASS')
print(f'  -> duong per-image dung ve CAU TRUC. Lech xac suat {_d:.1e} la nhieu so hoc'
      + ('' if _same_env else ' (moi truong khac nb16)') + '.')
print('     Moi so sanh trong nb17 dung CHUNG mot PIMG nen nhieu nay triet tieu khi so cap.')

# ---------------- T1.2 / F2 ----------------
R_TF = {t: eval_full(pool(PIMG[t]), f'transform {t}') for t in TRANSFORMS}
print()
print(f'=== T1.2 / F2 · ABLATION TRANSFORM (model {REF_MODEL}, pooling 0.7/0.3) ===')
rows = []
for t in TRANSFORMS:
    r = R_TF[t]
    row = dict(transform=t, **{g: r['point'][g] for g in GRP})
    if t != 'center518':
        for g in GRP:
            d, lo, hi, ex = delta_ci(r, R_TF['center518'], g)
            row[f'd_{g}'] = d; row[f'excl0_{g}'] = ex
    rows.append(row)
T_TF = pd.DataFrame(rows)
print(T_TF[['transform']+list(GRP)].to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print()
print('  --- delta vs center518 (paired, cluster=subject) ---')
for t in TRANSFORMS[1:]:
    r = R_TF[t]
    for g in GRP:
        d, lo, hi, ex = delta_ci(r, R_TF['center518'], g)
        print(f'  {t:<10} {g:<8} {d:+.5f}  CI95 [{lo:+.5f}, {hi:+.5f}]  {"LOAI 0" if ex else "chua 0"}')
T_TF.to_csv(f'{SC_DIR}/metrics/transform_ablation.csv', index=False)
_pa = {t: R_TF[t]['per_ap'] for t in TRANSFORMS}
print()
print('  --- AP tung nhan Rare-3 (noi gia thuyet ngoai vi song) ---')
print(pd.DataFrame({'label': [LABELS[j] for j in RARE3_IDX],
                    **{t: [_pa[t][j] for j in RARE3_IDX] for t in TRANSFORMS}}
                   ).to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print()
print('  NHAC: day chi la LAN CAT THU HAI (vien 21px). Lan cat thu nhat nam o 04a va')
print('  lon hon nhieu — chi 19_fov_ablation (F1/F3) moi cham toi. Ket qua am o day')
print('  KHONG bac bo gia thuyet nut that ngoai vi.')

STORE 559 -> MODEL 518 | OFF 41 | center offset 20
  gate: slice[20:538] == T.functional.center_crop  bit-for-bit  OK
per-image <- cache

=== MOI TRUONG: nb16 vs bay gio ===
  GPU   nb16 NVIDIA RTX PRO 6000 Blackwell Server Edition  |  bay gio NVIDIA A100-SXM4-80GB
  torch nb16 2.11.0+cu128  |  bay gio 2.11.0+cu128
  -> moi truong KHAC  => lech ~1e-4 tren xac suat la BINH THUONG

=== DIAGNOSTIC (KHONG phai cong): |pool(per-image) - R0_ep2.npy| ===
  max 1.471e-03 | trung vi 2.369e-05 | p99 1.845e-04

=== COÅNG A - Spearman tung nhan (>= 0.999) ===
  Atelectas 1.00000 | Cardiomeg 1.00000 | Consolida 1.00000 | Edema 1.00000 | Enlarged_ 1.00000 | Fracture 1.00000 | Lung_Lesi 1.00000 | Lung_Opac 1.00000 | No_Findin 1.00000 | Pleural_E 1.00000 | Pleural_O 1.00000 | Pneumonia 1.00000 | Pneumotho 1.00000
  thap nhat: Lung_Lesion 1.000000  PASS

=== COÅNG B - AP tai tao vs AP nb16 DA LOG (<= 2e-4) ===
  macro    tai tao 0.436471 | nb16 0.436463 | lech +0.000008
  rare3    tai tao 0.146719 | nb

## C6 · **V1 / V2 / V3** — multi-view *(CPU thuần)*

`pool_study` của nb11–16 cố định ba lựa chọn chưa bao giờ được kiểm:

1. `W_F = 0.7`, `W_L = 0.3` — **chung cho cả 13 nhãn**
2. **mean** trong mỗi view — một tổn thương chỉ hiện trên một phim thì mean pha loãng nó
3. Không có tham số nào học từ dữ liệu

| | thử gì | tham số |
|---|---|---|
| `V1` | `W_F` **theo từng nhãn**, lưới `[0, 1]` | 13 |
| `V2` | aggregator trong view: `mean` / `max` / `logsumexp` | 1 (rời rạc) |
| `V3` | **pooler học được**: logistic mỗi nhãn trên `[mean_F, max_F, mean_L, max_L, has_F, has_L, n_F, n_L]` | 8 × 13 |

`V3` bao trùm `V1` và `V2` — nếu nó không hơn thì pooling **không** phải nút thắt, và không cần
đụng tới attention bên trong model. Tiết kiệm được cả một nhánh GPU.

**Tất cả đều cross-fit 5-fold theo `subject_id`.** Số báo cáo là AP **out-of-fold**, không phải
AP của cấu hình tốt nhất trên chính val.

⚠ `W_F` chỉ ảnh hưởng study có **cả hai** view. `C4` in tỉ lệ đó — nếu nó nhỏ thì đòn bẩy `V1`
nhỏ theo, bất kể lưới tìm được gì.

In [14]:
PI = PIMG['center518']                       # giu transform hien hanh de tach bach hai hieu ung
LOGIT = lambda p: np.log(np.clip(p, 1e-7, 1-1e-7)/(1-np.clip(p, 1e-7, 1-1e-7)))

def ap_subset(ps, idx):
    """AP tren mot tap con study (dung cho fit trong cross-fit)."""
    out = np.full(C, np.nan)
    for j in range(C):
        y = Y_VAL[idx, j]
        if y.sum() == 0: continue
        o, gid, ng = _prep(ps[idx, j])
        out[j] = ap_grouped(y[o].astype('float64'), np.ones(len(idx)), gid, ng)
    return out

# ---------------- V1: W_F theo tung nhan, cross-fit ----------------
WF_GRID = np.linspace(0.0, 1.0, 21)
POOL_G = {w: pool(PI, wf=w, wl=1-w) for w in WF_GRID}          # cache 21 lan pool
def _v1_fit(tr):
    best = np.full(C, W_F)
    for j in range(C):
        sc = [ap_subset(POOL_G[w], tr)[j] for w in WF_GRID]
        if not np.all(np.isnan(sc)): best[j] = WF_GRID[int(np.nanargmax(sc))]
    return best
def _v1_apply(wf, te):
    out = np.empty((len(te), C), 'float32')
    for j in range(C): out[:, j] = POOL_G[wf[j]][te, j]
    return out
OOF_V1 = cross_fit_predict(_v1_fit, _v1_apply)
_wf_full = _v1_fit(np.arange(n_st))
print('=== V1 · W_F toi uu theo tung nhan (fit tren TOAN val — chi de XEM, khong bao cao) ===')
print(pd.DataFrame(dict(label=LABELS, tier=TIER, wf_opt=_wf_full,
                        wf_hien_hanh=W_F)).to_string(index=False))

# ---------------- V2: aggregator trong view ----------------
# Neu ro KHONG GIAN: 'mean' trong khong gian XAC SUAT la moc cua nb11-16.
# mean KHONG giao hoan voi logit nen mean_prob != mean_logit; max thi co.
# Ba bien the con lai deu o khong gian logit de nhat quan voi buoc gop cross-view.
POOL_H = {'mean_prob':  pool(PI,        how='mean'),      # = moc nb16
          'mean_logit': pool(LOGIT(PI), how='mean'),
          'max_logit':  pool(LOGIT(PI), how='max'),
          'lse_logit':  pool(LOGIT(PI), how='lse')}
def _v2_fit(tr):
    sc = {h: np.nanmean(ap_subset(POOL_H[h], tr)) for h in POOL_H}
    return max(sc, key=sc.get)
OOF_V2 = cross_fit_predict(_v2_fit, lambda h, te: POOL_H[h][te])

# ---------------- V3: pooler hoc duoc ----------------
from sklearn.linear_model import LogisticRegression
_mF, _hF = _agg(LOGIT(PI), IS_F, 'mean'); _xF, _ = _agg(LOGIT(PI), IS_F, 'max')
_mL, _hL = _agg(LOGIT(PI), IS_L, 'mean'); _xL, _ = _agg(LOGIT(PI), IS_L, 'max')
_nF = np.bincount(SIDI[IS_F], minlength=len(SID_VAL)).astype('float64')
_nL = np.bincount(SIDI[IS_L], minlength=len(SID_VAL)).astype('float64')
def _feat(j):
    return np.column_stack([np.nan_to_num(_mF[:, j]), np.nan_to_num(_xF[:, j]),
                            np.nan_to_num(_mL[:, j]), np.nan_to_num(_xL[:, j]),
                            _hF.astype('float64'), _hL.astype('float64'),
                            np.log1p(_nF), np.log1p(_nL)])
X3 = [_feat(j) for j in range(C)]
def _v3_fit(tr):
    ms = []
    for j in range(C):
        y = Y_VAL[tr, j]
        if y.sum() < 5 or y.sum() == len(tr): ms.append(None); continue
        ms.append(LogisticRegression(max_iter=2000, C=1.0).fit(X3[j][tr], y))
    return ms
def _v3_apply(ms, te):
    out = np.empty((len(te), C), 'float32')
    for j in range(C):
        out[:, j] = (ms[j].predict_proba(X3[j][te])[:, 1] if ms[j] is not None
                     else POOL_G[W_F][te, j])
    return out
OOF_V3 = cross_fit_predict(_v3_fit, _v3_apply)

# ---------------- so sanh ----------------
R_V0 = R_TF['center518']                                  # moc: 0.7/0.3, mean
R_V = dict(V1_wf_per_label=eval_full(OOF_V1, 'V1 W_F/nhan'),
           V2_aggregator=eval_full(OOF_V2, 'V2 aggregator'),
           V3_learned_pooler=eval_full(OOF_V3, 'V3 pooler hoc duoc'))
print()
print('=== V1/V2/V3 vs pooling hien hanh (0.7/0.3, mean) — TAT CA out-of-fold ===')
rows = []
for nm, r in R_V.items():
    row = dict(variant=nm)
    for g in GRP:
        d, lo, hi, ex = delta_ci(r, R_V0, g)
        row[g] = r['point'][g]; row[f'd_{g}'] = d; row[f'excl0_{g}'] = ex
    rows.append(row)
    print(f'  {nm:<20} ' + ' | '.join(f'{g} {r["point"][g]:.4f}' for g in GRP))
print()
print('  --- delta vs moc (paired, cluster=subject) ---')
print(f'  {"moc 0.7/0.3":<20} ' + ' | '.join(f'{g} {R_V0["point"][g]:.4f}' for g in GRP))
for nm, r in R_V.items():
    for g in ('macro', 'rare3', 'tail'):
        d, lo, hi, ex = delta_ci(r, R_V0, g)
        print(f'  {nm:<20} {g:<7} {d:+.5f}  CI95 [{lo:+.5f}, {hi:+.5f}]  {"LOAI 0" if ex else "chua 0"}')
T_V = pd.DataFrame(rows); T_V.to_csv(f'{SC_DIR}/metrics/multiview.csv', index=False)
json.dump(dict(wf_opt_full_val={l: float(w) for l, w in zip(LABELS, _wf_full)},
               baseline=R_V0['point'], variants={k: v['point'] for k, v in R_V.items()},
               n_study_both_views=int(_both), frac_both_views=float(_both/len(SID_VAL)),
               note='moi bien the deu cross-fit 5-fold theo subject_id; AP la out-of-fold'),
          open(f'{SC_DIR}/metrics/multiview.json', 'w'), indent=1)

=== V1 · W_F toi uu theo tung nhan (fit tren TOAN val — chi de XEM, khong bao cao) ===
                     label   tier  wf_opt  wf_hien_hanh
               Atelectasis   head    0.75           0.7
              Cardiomegaly   head    0.75           0.7
             Consolidation medium    0.75           0.7
                     Edema   head    0.80           0.7
Enlarged_Cardiomediastinum medium    0.65           0.7
                  Fracture medium    1.00           0.7
               Lung_Lesion medium    0.95           0.7
              Lung_Opacity   head    0.80           0.7
                No_Finding   head    0.65           0.7
          Pleural_Effusion   head    0.50           0.7
             Pleural_Other   tail    0.90           0.7
                 Pneumonia   head    0.70           0.7
              Pneumothorax medium    0.85           0.7

=== V1/V2/V3 vs pooling hien hanh (0.7/0.3, mean) — TAT CA out-of-fold ===
  V1_wf_per_label      macro 0.4366 | rare3 0.1480 | 

## C7 · Tổng kết + cổng quyết định

Cổng đặt **trước** khi đọc số, để một kết quả nhỏ không bị diễn giải thành tín hiệu.

| nhánh | tiếp nếu | dừng nếu |
|---|---|---|
| `T2` head-only retrain | `T1` không nâng được single-model, và `V3` cho thấy pooling **không** phải nút thắt | `T1`/`V` đã đủ |
| `T3` partial FT | `T2` chỉ ra head **không** phải nút thắt | `T2` đã đủ |
| `V` sâu hơn (attention trong model) | `V3` hơn mốc **và** CI loại 0 | `V3` không hơn `V1` ⇒ pooling không phải nút thắt |
| `F3` letterbox | quyết ở `19_fov_ablation` theo `F1`, **không** theo `C5` | — |

Nhắc lại nguyên tắc đã khoá: **ensemble là một dòng riêng**, không trộn vào con số single-model.

In [15]:
print('=== TONG KET nb17 ===')
print()
print('--- SINGLE-MODEL (con so CHINH) ---')
_single = [('nb16 R0 ep2 (moc)', R_V0)] + \
          [(f'transform {t}', R_TF[t]) for t in TRANSFORMS if t != 'center518'] + \
          [(nm, r) for nm, r in R_V.items()]
best = max(_single, key=lambda kv: kv[1]['point']['macro'])
for nm, r in _single:
    mark = '  <-- macro cao nhat' if nm == best[0] else ''
    print(f'  {nm:<26} ' + ' | '.join(f'{g} {r["point"][g]:.4f}' for g in GRP) + mark)
print()
print('--- ENSEMBLE (dong RIENG) ---')
print(f'  {"".join(ENS_PRIMARY[:1])+f" +{len(ENS_PRIMARY)-1} ckpt ({ENS_RULE})":<26} '
      + ' | '.join(f'{g} {R_ENS["point"][g]:.4f}' for g in GRP))
print()
print('=== COÅNG QUYET DINH ===')
_d_best, _lo, _hi, _ex = delta_ci(best[1], R_V0, 'macro')
print(f'  bien the single tot nhat: {best[0]}  dMacro {_d_best:+.5f} '
      f'CI [{_lo:+.5f}, {_hi:+.5f}] {"LOAI 0" if _ex else "CHUA 0"}')
_v3d, _v3lo, _v3hi, _v3ex = delta_ci(R_V['V3_learned_pooler'], R_V0, 'macro')
_v1d = delta_ci(R_V['V1_wf_per_label'], R_V0, 'macro')[0]
print()
print(f'  V3 - moc      dMacro {_v3d:+.5f} CI [{_v3lo:+.5f}, {_v3hi:+.5f}] '
      f'{"LOAI 0" if _v3ex else "chua 0"}')
print(f'  V3 - V1       dMacro {_v3d-_v1d:+.5f}')
if not _v3ex:
    print('  -> POOLING KHONG phai nut that. KHONG dau tu attention multi-view trong model.')
else:
    print('  -> pooling CO du dia. Xem xet attention multi-view trong model (uu tien 3).')
print()
if _ex and _d_best > 0:
    print(f'  -> T1/V da nang duoc single-model {_d_best:+.4f} macro voi CI loai 0.')
    print('     Chua can T2/T3. Chot bien the nay lam moc moi roi sang 18/19.')
else:
    print('  -> T1/V KHONG nang duoc single-model mot cach do duoc.')
    print('     -> leo len T2 (head-only retrain, ~4 h GPU). Neu T2 cung khong,')
    print('        thi head KHONG phai nut that -> T3 (partial FT lich da sua, ~3 h).')
print()
print('  LUU Y: moi so o day la out-of-fold hoac khong co tham so fit. Nhung viec CHON')
print('  bien the tot nhat de bao cao van la mot lua chon — neu chot mot bien the moi lam moc,')
print('  ghi ro no duoc chon tren VAL, va moi so sanh ve sau phai dung moc do NHAT QUAN.')
json.dump(dict(single={nm: r['point'] for nm, r in _single},
               ensemble={f'{ENS_RULE}:{"+".join(ENS_PRIMARY)}': R_ENS['point']},
               best_single=best[0], best_single_delta_macro=_d_best,
               best_single_ci=[_lo, _hi], best_single_excl0=bool(_ex),
               v3_minus_baseline_macro=_v3d, v3_excl0=bool(_v3ex),
               pooling_is_bottleneck=bool(_v3ex)),
          open(f'{SC_DIR}/metrics/summary.json', 'w'), indent=1)
print()
print(f'-> {SC_DIR}/metrics/summary.json')

=== TONG KET nb17 ===

--- SINGLE-MODEL (con so CHINH) ---
  nb16 R0 ep2 (moc)          macro 0.4365 | rare3 0.1467 | nontgt 0.5234 | tail 0.1191 | worst3 0.1339
  transform resize518        macro 0.4371 | rare3 0.1469 | nontgt 0.5242 | tail 0.1204 | worst3 0.1347  <-- macro cao nhat
  V1_wf_per_label            macro 0.4366 | rare3 0.1480 | nontgt 0.5232 | tail 0.1176 | worst3 0.1358
  V2_aggregator              macro 0.4368 | rare3 0.1470 | nontgt 0.5237 | tail 0.1192 | worst3 0.1339
  V3_learned_pooler          macro 0.4369 | rare3 0.1484 | nontgt 0.5235 | tail 0.1169 | worst3 0.1353

--- ENSEMBLE (dong RIENG) ---
  EP0 +2 ckpt (mean_prob)    macro 0.4376 | rare3 0.1496 | nontgt 0.5240 | tail 0.1238 | worst3 0.1373

=== COÅNG QUYET DINH ===
  bien the single tot nhat: transform resize518  dMacro +0.00066 CI [-0.00052, +0.00174] CHUA 0

  V3 - moc      dMacro +0.00047 CI [-0.00096, +0.00199] chua 0
  V3 - V1       dMacro +0.00030
  -> POOLING KHONG phai nut that. KHONG dau tu attenti